
4. Add a second target (staging or prod) to your databricks.yml with a different workspace host and run_as service principal, and deploy to it. 

---->>

![image_1790674667630.png](./image_1790674667630.png "image_1790674667630.png")

![image_1790674705045.png](./image_1790674705045.png "image_1790674705045.png")



5. Configure M2M (service principal) authentication for the CLI and use it instead of your personal U2M login for a deploy command. 

this is done on hands on -->>


![image_1791360727971.png](./image_1791360727971.png "image_1791360727971.png")


6. Write a GitHub Actions workflow that runs databricks bundle validate on every pull request, without deploying anything. 

-->>
here the github action code -->>

```
# Name of the GitHub Actions workflow
name: 'Deploy DB Bundle - Production'

concurrency: 1

# Trigger the workflow whenever code is pushed to the main branch.
# A merged Pull Request into main creates a push event, so this
# workflow will run automatically after the PR is merged.
on:
  push:
    branches:
      - main
    paths:
      - 'Bakehouse_analytics/**'

# Define the jobs that will run as part of this workflow
jobs:

  # Job responsible for deploying the Databricks bundle to production
  deploy-prod:
    name: 'Deploy Bundle'
    runs-on: ubuntu-latest

    # Permissions required by the workflow
    permissions:
      # Allow the workflow to read the repository contents
      contents: read

    # Steps executed sequentially
    steps:

      # Checkout the latest code from the main branch
      - name: Checkout repository
        uses: actions/checkout@v4

      # Install the Databricks CLI on the GitHub Actions runner
      - name: Install Databricks CLI
        uses: databricks/setup-cli@main 

      # Validate the Databricks Asset Bundle configuration
      # before attempting to deploy it.
      #
      # The "-t prod" option tells Databricks CLI to use
      # the "prod" target defined in databricks.yml.
      - name: Validate Databricks Bundle
        run: databricks bundle validate -t prod
        working-directory: Bakehouse_analytics
        env:
          # Databricks workspace URL stored as a GitHub Secret
          
          # DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}

          # Databricks authentication token stored as a GitHub Secret
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}

      # Deploy the validated Databricks Asset Bundle
      # to the production environment.
      - name: Deploy Databricks Bundle to Production
        run: databricks bundle deploy -t prod
        working-directory: Bakehouse_analytics
        env:
          # Production Databricks workspace URL
          
          # DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}

          # Authentication token used to authenticate with Databricks
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}

```
![image_1791373010398.png](./image_1791373010398.png "image_1791373010398.png")